# QA Service


### Instalar as bibliotecas necessárias

In [ ]:
# !pip install qdrant-client langchain-qdrant pandas

### Importar as bibliotecas

In [1]:
# Manipulação 
import os
import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field

# Qdrant
from qdrant_client import models, QdrantClient

# Langchain
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain.memory import ConversationBufferMemory
from langchain.schema.runnable import RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents.format_scratchpad import format_to_openai_functions
from langchain.agents.output_parsers import OpenAIFunctionsAgentOutputParser
from langchain_core.utils.function_calling import convert_to_openai_function

### Configurar variáveis de ambiente

In [2]:
# Carregar as variáveis do .env
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API")
qdrant_api = os.getenv("QDRANT_API_KEY")
qdrant_url = os.getenv("QDRANT_URL")

### Inicializar os clientes

In [3]:
# Qdrant para teste
client = QdrantClient(
    url=qdrant_url,
    api_key=qdrant_api
)

In [4]:
# Embeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

### Listar coleções presentes no cluster

In [5]:
collections = client.get_collections().collections
print("Coleções disponíveis:", [c.name for c in collections])

Coleções disponíveis: ['ppgegc_qa_method', 'PPGEGC_UFSC', 'santa_catarina_jurisprudence']


### Conectar a uma coleção específica

In [6]:
collection_name = "PPGEGC_UFSC"

vector_store = QdrantVectorStore.from_existing_collection(
    url=qdrant_url,
    api_key=qdrant_api,
    collection_name=collection_name,
    embedding=embeddings
)


### TESTES

1. Obtendo todos os points e colocando-os em um df

In [7]:
# Obter a quantidade de documentos na coleção
n_docs = vector_store.client.get_collection(vector_store.collection_name).points_count
print(f"Quantidade de documentos na coleção {collection_name}: {n_docs}")

Quantidade de documentos na coleção PPGEGC_UFSC: 150


In [8]:
# Obter todos os documentos da coleção de 100 em 100
batch_size = 100
all_documents = []
offset = 0

while True:
    results = vector_store.client.scroll(
        collection_name=vector_store.collection_name,
        offset=offset,
        limit=batch_size,
        with_payload=True,
        with_vectors=True
    )
    
    all_documents.extend(results[0])
    
    if len(results[0]) < batch_size:
        break
    
    offset += batch_size

In [9]:
print(f"Total de documentos obtidos: {len(all_documents)}")

Total de documentos obtidos: 151


In [10]:
print(all_documents)

[Record(id=1, payload={'autor': 'jorge ivan hmeljevski', 'titulo': 'modelo para sistemas de supervisão de mercado baseados em conhecimento', 'tipo': 'tese', 'area_de_concentracao': 'engenharia do conhecimento', 'ano_de_publicacao': 2021, 'local': 'florianópolis', 'orientador': 'jose leomar todesco', 'coorientador': 'alexandre leopoldo goncalves', 'resumo': 'a confiança na higidez dos mercados de capitais é promovida por organizações reguladoras que estabelecem as regras de atuação nesses mercados, supervisionam seu funcionamento e sancionam aqueles que não as cumprem. nestas organizações, especialistas em supervisão trabalham visando detectar, investigar pot enciais irregularidades e punir infratores. e stes especialistas  lidam com grande quantidade de dados, estruturados e não -estruturados, provenientes de diversas fontes, internas e externas à organização reguladora. trata -se de um trabalho complexo, incerto e que demanda conhecimento especializado, portanto, um típico trabalho in

In [11]:
data = [{
    "embeddings": doc.vector,
    **doc.payload
    } for doc in all_documents]

df = pd.DataFrame(data)

In [12]:
df.head()

,embeddings,autor,titulo,tipo,area_de_concentracao,ano_de_publicacao,local,orientador,coorientador,resumo,palavras_chave,abstract,keywords,introducao_contextualizacao,introducao_problematica,introducao_ineditismo,introducao_contribuicao,conclusao
0,"[0.022624256, 0.063604906, 0.025148261, 0.0164...",jorge ivan hmeljevski,modelo para sistemas de supervisão de mercado ...,tese,engenharia do conhecimento,2021,florianópolis,jose leomar todesco,alexandre leopoldo goncalves,a confiança na higidez dos mercados de capitai...,"['mercado de capitais', 'mercado de valores mo...",the confidence in the integrity of capital mar...,"['capital market', 'securities market', 'marke...",o mercado de capitais é fundamental para o cre...,"os ssm, de maneira geral, usam os dado s prove...","de maneira inédita, portanto, este trabalho pa...","a contribuição desta pesquisa, portanto, está ...",o modelo elaborado nest a pesquisa envolveu a ...
1,"[-0.00624535, 0.06508915, 0.0091159195, -0.000...",ivam galvão filho,fractus: aplicativo para aprendizagem de frações,dissertação,engenharia do conhecimento,2022,florianópolis,vania ribas ulbricht,elisa maria pivetta,o objetivo principal desta pesquisa foi o dese...,"['objetos de aprendizagem.', 'frações.', 'apli...",the main objective of this research was the de...,"['learning objects.', 'fractions.', 'app for l...",o sistema educacional brasileiro passa por uma...,a deficiência na aprendizagem nas escolas de e...,,,o trabalho realizado pela organização todos pe...
2,"[0.024945587, 0.04501195, 0.016351996, 0.00542...",márcio crescencio,modelo de uma rede colaborativa suportada por ...,tese,engenharia do conhecimento,2022,florianópolis,alexandre augusto biz,jose leomar todesco,a convergência entre o turismo e a cultura atr...,"['sítios de patrimônio mundial', 'gestão do tu...",the convergence between tourism and culture th...,"['world heritage sites', 'tourism management',...",o turismo se tornou uma das maiores indústrias...,"a convergência entre o turismo e a cultura, at...",o reconhecimento de pm atrai turistas adiciona...,esses elementos de ligação do desenvolvimento ...,esta tese identificou que o turismo possui um ...
3,"[0.0503207, 0.051356107, 0.0053168065, 0.01170...",roseli honorio,modelo conceitual de governança de dados como ...,dissertação,engenharia do conhecimento,2022,florianópolis,joao artur de souza,patricia de sa freire,a humanidade passou por transformações e revol...,"['governança de dados', 'governança do conheci...",humanity has undergone transformations and rev...,"['data governance', 'knowledge governance', 'f...",a sociedade está atravessando um momento de mu...,é nesse contexto que a governanç a do conhecim...,,,"na ind ústria 4.0 e sociedade 5.0, as organiza..."
4,"[-0.0012473083, 0.044789646, -0.03970279, -0.0...",josé tadeu silva,análise da contribuição da engenharia do conhe...,dissertação,engenharia do conhecimento,2022,florianópolis,fernando alvaro ostuni gauthier,marcelo macedo,a presente dissertação aborda as questões emer...,"['comércio eletrônico', 'modelo de análise', '...",the present dissertation addresses the emergin...,"['e-commerce', 'analysis model', 'knowledge en...","de acordo com lemos (2003), sob qualquer aspec...","para vilaça e araújo (2016), o conhecimento so...",,,a partir do objetivo geral de analisar as cont...


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 151 entries, 0 to 150
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   embeddings                   151 non-null    object
 1   autor                        151 non-null    object
 2   titulo                       151 non-null    object
 3   tipo                         151 non-null    object
 4   area_de_concentracao         151 non-null    object
 5   ano_de_publicacao            151 non-null    int64 
 6   local                        151 non-null    object
 7   orientador                   151 non-null    object
 8   coorientador                 151 non-null    object
 9   resumo                       151 non-null    object
 10  palavras_chave               151 non-null    object
 11  abstract                     151 non-null    object
 12  keywords                     151 non-null    object
 13  introducao_contextualizacao  151 no

2. Teste de busca por área do conhecimento simples sem filtro

In [13]:
query = "Área de conhecimento: Engenharia do Conhecimento"

results = vector_store.client.search(
    collection_name=vector_store.collection_name,
    query_vector=embeddings.embed_query(query),
    limit=15,
)

for res in results:
    print(res.payload, "score:", res.score)

{'autor': 'josé tadeu silva', 'titulo': 'análise da contribuição da engenharia do conhecimento ao comércio eletrônico', 'tipo': 'dissertação', 'area_de_concentracao': 'engenharia do conhecimento', 'ano_de_publicacao': 2022, 'local': 'florianópolis', 'orientador': 'fernando alvaro ostuni gauthier', 'coorientador': 'marcelo macedo', 'resumo': 'a presente dissertação aborda as questões emergentes do comércio eletrônico como campo fértil para aplicação de métodos e técnicas da engenharia do conhecimento no seu aperfeiçoamento. as transformações necessárias para enfrentamento dos desafios exigidos por um crescimento vertiginoso requerem cada vez mais conhecimento do negócio. para isso, formulou-se como problema de pesquisa as contribuições que a engenharia do conhecimento é capaz de fornecer. com a pesquisa efetuada nas devidas bases teóricas, adequou-se um modelo de análise, originalmente utilizado na gestão do conhecimento, como possível indicativo de ferramentas da engenharia do conhecim

In [14]:
for res in results:
    print(res.payload.get("area_de_concentracao"), "score:", res.score)

engenharia do conhecimento score: 0.5879588
engenharia do conhecimento score: 0.583552
gestão do conhecimento score: 0.58297014
gestão do conhecimento score: 0.55421275
engenharia do conhecimento score: 0.5493218
engenharia do conhecimento score: 0.54925525
gestão do conhecimento score: 0.54793245
gestão do conhecimento score: 0.544219
engenharia do conhecimento score: 0.53917694
engenharia do conhecimento score: 0.53753936
gestão do conhecimento score: 0.53491074
engenharia do conhecimento score: 0.5258687
mídia do conhecimento score: 0.52399397
mídia do conhecimento score: 0.52335584
engenharia do conhecimento score: 0.52136886


In [15]:
query = "Área de conhecimento: Gestão do Conhecimento"

results = vector_store.client.query_points(
    collection_name=vector_store.collection_name,
    query=embeddings.embed_query(query),
    limit=15,
).points

for res in results:
    print(res.payload.get("area_de_concentracao"), "score:", res.score)

gestão do conhecimento score: 0.6204455
gestão do conhecimento score: 0.60022986
engenharia do conhecimento score: 0.5960252
gestão do conhecimento score: 0.5832628
gestão do conhecimento score: 0.5761869
mídia do conhecimento score: 0.5756919
gestão do conhecimento score: 0.57392406
gestão do conhecimento score: 0.5698453
engenharia do conhecimento score: 0.5648972
gestão do conhecimento score: 0.56057644
engenharia do conhecimento score: 0.5577202
mídia do conhecimento score: 0.55411434
gestão do conhecimento score: 0.55290693
gestão do conhecimento score: 0.5416412
mídia do conhecimento score: 0.53956395


In [ ]:
query = "Teses e Dissertações orientados pelo Professor Alexandre Leopoldo Gonçalves"

results = vector_store.client.query_points(
    collection_name=vector_store.collection_name,
    query=embeddings.embed_query(query),
    limit=5,
).points

for res in results:
    print("score:", res.score, res)

score: 0.44464892 id=104 version=2 score=0.44464892 payload={'autor': 'Sergio Luiz Gargioni', 'titulo': 'MODELO OPERACIONAL DE EDUCAÇÃO CONTINUADA PARA PROFISSIONAIS DE ENGENHARIA BASEADO EM COMPETÊNCIAS DIGITAIS PARA ATENDER A DESAFIOS E OPORTUNIDADES DA TRANSFORMAÇÃO DIGITAL', 'tipo': 'tese', 'area_de_concentracao': 'Gestão do Conhecimento', 'ano_de_publicacao': 2023, 'local': 'Florianópolis – SC', 'orientador': 'Professor Neri dos Santos, Dr. Ing.', 'coorientador': 'Professor Gregório Jean Varvakis Rados, Dr. Ing.', 'resumo': 'Este trabalho de tese apresenta um modelo operacional de educação continuada para profissionais de engenharia atuantes no mercado e para estudantes regulares em final do curso de graduação, como extensão ou disciplinas optativas, com foco na abordagem de Transformação Digital e baseado nos conceitos contemporâneos e atuais de aprendizagem por competências digitais, exigidas pelo mundo empresarial. Isso será alcançado por meio da oferta de um programa de educaç

2. Teste de busca simples com filtro

In [17]:
query = "Área de conhecimento: Gestão do Conhecimento"

results = vector_store.client.query_points(
    collection_name=vector_store.collection_name,
    query=embeddings.embed_query(query),
    query_filter=models.Filter(
        must=[models.FieldCondition(key="area_de_concentracao", match=models.MatchValue(
            value="gestão do conhecimento"
        ))]
    ),
    limit=15,
).points

for res in results:
    print(res.payload.get("area_de_concentracao"), "score:", res.score)

gestão do conhecimento score: 0.6204455
gestão do conhecimento score: 0.60022986
gestão do conhecimento score: 0.5832628
gestão do conhecimento score: 0.5761869
gestão do conhecimento score: 0.57392406
gestão do conhecimento score: 0.5698453
gestão do conhecimento score: 0.56057644
gestão do conhecimento score: 0.55290693
gestão do conhecimento score: 0.5416412
gestão do conhecimento score: 0.53431123
gestão do conhecimento score: 0.5281006
gestão do conhecimento score: 0.51592934
gestão do conhecimento score: 0.5148233
gestão do conhecimento score: 0.5062214
gestão do conhecimento score: 0.50526327


In [32]:
query = "Inteligência Artificial"

results = vector_store.client.query_points(
    collection_name=vector_store.collection_name,
    query=embeddings.embed_query(query),
    query_filter=models.Filter(
        must=[
            models.FieldCondition(
                key="orientador",
                match=models.MatchText(
                    text="alexandre leopoldo"
                    ),
                ),
            ],
        should=[
                models.FieldCondition(
                    key="ano_de_publicacao",
                    match=models.MatchValue(
                        value=2023
                        ),
                    ),
                models.FieldCondition(
                    key="ano_de_publicacao",
                    match=models.MatchValue(
                        value=2022
                        ),
                    ),
                models.FieldCondition(
                    key="ano_de_publicacao",
                    match=models.MatchValue(
                        value=2021
                        ),
                    ),
                ]
    ),
    limit=15,
).points

for res in results:
    print("score:", res.score, res)

score: 0.47285628 id=38 version=0 score=0.47285628 payload={'autor': 'letícia silveira artese', 'titulo': 'modelo de descoberta de conhecimento em texto para detecção de sinais fracos para tecnologias emergentes', 'tipo': 'tese', 'area_de_concentracao': 'engenharia do conhecimento', 'ano_de_publicacao': 2023, 'local': 'florianópolis', 'orientador': 'alexandre leopoldo goncalves', 'coorientador': 'jose leomar todesco', 'resumo': 'sinais fracos são fragmentos de informação que a princípio podem parecer vagos e desconexos, mas que atuam como indicadores de um evento futuro e potenciais questões emergentes. em particular, o domínio tecnológico se destaca no interesse em antecipar informações, visto a estreita relação entre avanço tecnológico e vantagem competitiva. nessa perspectiva, sinais fracos se mostram uma informação relevante para o planejamento estratégico. o contexto atual de desenvolvimento acelerado enaltece a necessidade de uma abordagem que considere o conceito de incerteza no

---
## Configurações para o QA
---

### Configurando Basic Tool

In [78]:
from typing import Optional, List

class QdrantQueryInput(BaseModel):
    query: str = Field(..., description="Pergunta do usuário completa sem alterações")
    query_filter: str = Field(..., description="Pergunta do usuário reduzida que servirá para busca semântica para documentos semelhantes (pode ser vazia \"\")")
    autor: Optional[List[str]] = Field(None, description="Nome do autor ou autores (sempre minusculo, sem acentos e sem caracteres especiais como: ç)")
    tipo: Optional[List[str]] = Field(None, description="Tipo é referente a se o usuário pedir tese e/ou dissertação)")
    area_de_concentracao: Optional[List[str]] = Field(None, description="Área ou áreas de concentração podem ser: engenharia do conhecimento, gestão do conhecimento e mídia do conhecimento")
    ano_de_publicacao: Optional[List[int]] = Field(None, description="Ano de publicação do documento")
    orientador: Optional[List[str]] = Field(None, description="Nome do orientador ou orientadores")
    coorientador: Optional[List[str]] = Field(None, description="Nome do coorientador ou coorientadores")

@tool(args_schema=QdrantQueryInput)
def qdrant_query_tool(
    query: str,
    query_filter: str,
    autor: Optional[List[str]] = None,
    tipo: Optional[List[str]] = None,
    area_de_concentracao: Optional[List[str]] = None,
    ano_de_publicacao: Optional[List[int]] = None,
    orientador: Optional[List[str]] = None,
    coorientador: Optional[List[str]] = None,
) -> str:
    """Useful to perform a query in Qdrant with multiple additional filters."""
    
    query_embedding = embeddings.embed_query(query_filter)
    must_filters = []
    should_filters = []

    # Processa cada filtro e decide se vai para must ou should
    def add_filter(key, values, is_text=True):
        if values:
            conditions = [
                models.FieldCondition(
                    key=key,
                    match=models.MatchText(text=value) if is_text else models.MatchValue(value=value)
                )
                for value in values
            ]
            if len(conditions) > 1:
                should_filters.extend(conditions)
            else:
                must_filters.append(conditions[0])

    add_filter("autor", autor, is_text=True)
    add_filter("tipo", tipo, is_text=True)
    add_filter("area_de_concentracao", area_de_concentracao, is_text=True)
    add_filter("ano_de_publicacao", ano_de_publicacao, is_text=False)
    add_filter("orientador", orientador, is_text=True)
    add_filter("coorientador", coorientador, is_text=True)

    # Combina os filtros must e should
    qdrant_filter = models.Filter(
        must=must_filters,
        should=should_filters
    ) if must_filters or should_filters else None

    # Realiza a consulta ao Qdrant
    results = vector_store.client.query_points(
        collection_name=vector_store.collection_name,
        query=query_embedding,
        query_filter=qdrant_filter,
        limit=10,
    ).points

    return query + str(results)


In [79]:
tools = [qdrant_query_tool]

### Configurando o Model para a OpenAI

In [80]:
functions = [convert_to_openai_function(f) for f in tools]
model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
).bind(functions=functions)

### Configurando o prompt

In [67]:
template = """
    Você é o assistente virtual do QA Thesis and Dissertation.
    
    Sua função é responder somente sobre perguntas relacionadas a teses e dissertações de programas de pós-graduação.
    Para isso use a tool 'qdrant_query_tool' para fazer as consultas ao banco de dados se necessário, passando os valores todos em caractere minúsculos.
    Quando acessar essa tool envie o input completo e sem alterações como 'query' e em 'query_filter' apenas o texto pertinente para a busca semelhante, opcionalmente:
    
    autor: Optional[List[str]] = None,
    tipo: Optional[List[str]] = None,
    area_de_concentracao: Optional[List[str]] = None,
    ano_de_publicacao: Optional[List[int]] = None,
    orientador: Optional[List[str]] = None,
    coorientador: Optional[List[str]] = None,
    
    caso apareçam na query, ou tenha algo que referencie esses campos.
    
    OBS: a tool 'qdrant_query_tool' passe sempre caracteres minusculos e sem acentos, também não utilize (ç) ou outros caracteres especiais. 
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", template),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

In [81]:
agent_chain = RunnablePassthrough.assign(
    agent_scratchpad= lambda x: format_to_openai_functions(x["intermediate_steps"])
) | prompt | model | OpenAIFunctionsAgentOutputParser()

In [61]:
memory = ConversationBufferMemory(return_messages=True,memory_key="chat_history")

In [62]:
agent_executor = AgentExecutor(agent=agent_chain, tools=tools, verbose=True, memory=memory)

In [51]:
result = agent_executor.invoke({"input": "Realize uma síntese dos principais assuntos relacionado ao assunto machine learning e inteligência atificial na área de engenharia do conhecimento e considerando os anos de 2021 até 2023"})



> Entering new AgentExecutor chain...

Invoking: `qdrant_query_tool` with `{'query': 'sintese dos principais assuntos relacionados ao assunto machine learning e inteligencia artificial na area de engenharia do conhecimento entre 2021 e 2023', 'query_filter': 'machine learning inteligencia artificial engenharia do conhecimento', 'ano_de_publicacao': [2021, 2022, 2023]}`


sintese dos principais assuntos relacionados ao assunto machine learning e inteligencia artificial na area de engenharia do conhecimento entre 2021 e 2023[ScoredPoint(id=19, version=0, score=0.60403585, payload={'autor': 'kaster, gerson bovi', 'titulo': 'framework conceitual baseado em aprendizagem de máquina supervisionada para concepção de sistemas judiciais de agentes inteligentes', 'tipo': 'dissertação', 'area_de_concentracao': 'engenharia do conhecimento', 'ano_de_publicacao': 2021, 'local': 'florianópolis', 'orientador': 'aires jose rover', 'coorientador': 'nao disponivel', 'resumo': 'atualmente existem várias 

In [52]:
result['output']

'Nos últimos anos, entre 2021 e 2023, diversos estudos na área de Engenharia do Conhecimento têm explorado a interseção entre Machine Learning (ML) e Inteligência Artificial (IA). Aqui estão alguns dos principais tópicos abordados:\n\n1. **Frameworks e Modelos de Aplicação**:\n   - Um estudo de Gerson Bovi Kaster (2021) propôs um framework conceitual baseado em aprendizagem de máquina supervisionada para a concepção de sistemas judiciais de agentes inteligentes, focando na automação de tarefas em sistemas de informação na área jurídica. O trabalho enfatiza a importância de aplicar técnicas de IA em sistemas já existentes e a necessidade de metodologias adequadas para a implementação.\n\n2. **Assistentes Virtuais e Chatbots**:\n   - A dissertação de Lodacir Rodrigo Silva da Rosa (2023) desenvolveu um assistente virtual baseado em conhecimento para atender demandas administrativas de alunos de pós-graduação. O estudo utilizou técnicas de modelagem do conhecimento e ontologias, destacando

In [82]:
agent_executor_not_memory = AgentExecutor(agent=agent_chain, tools=tools, verbose=True)

In [64]:
result_not_memory = agent_executor_not_memory.invoke({"input": "Realize uma síntese dos principais assuntos relacionado ao assunto machine learning e inteligência atificial na área de engenharia do conhecimento e considerando os anos de 2021 até 2023"})



> Entering new AgentExecutor chain...

Invoking: `qdrant_query_tool` with `{'query': 'sintese dos principais assuntos relacionados ao assunto machine learning e inteligencia artificial na area de engenharia do conhecimento considerando os anos de 2021 até 2023', 'query_filter': 'machine learning inteligencia artificial engenharia do conhecimento', 'ano_de_publicacao': [2021, 2022, 2023]}`


sintese dos principais assuntos relacionados ao assunto machine learning e inteligencia artificial na area de engenharia do conhecimento considerando os anos de 2021 até 2023[ScoredPoint(id=19, version=0, score=0.60403585, payload={'autor': 'kaster, gerson bovi', 'titulo': 'framework conceitual baseado em aprendizagem de máquina supervisionada para concepção de sistemas judiciais de agentes inteligentes', 'tipo': 'dissertação', 'area_de_concentracao': 'engenharia do conhecimento', 'ano_de_publicacao': 2021, 'local': 'florianópolis', 'orientador': 'aires jose rover', 'coorientador': 'nao disponivel

In [65]:
result_not_memory['output']

'Nos últimos anos, de 2021 a 2023, a interseção entre machine learning, inteligência artificial e engenharia do conhecimento tem sido um tema de crescente relevância em diversas pesquisas acadêmicas. Abaixo, apresento uma síntese dos principais assuntos abordados em teses e dissertações relacionadas a esse tema:\n\n1. **Frameworks e Modelos de Aplicação**:\n   - **Desenvolvimento de Frameworks**: Pesquisas têm se concentrado na criação de frameworks que integram técnicas de machine learning e inteligência artificial para a construção de sistemas de conhecimento. Um exemplo é a dissertação de Gerson Bovi Kaster (2021), que propõe um framework conceitual baseado em aprendizagem de máquina supervisionada para sistemas judiciais, visando automatizar tarefas e melhorar a eficiência do judiciário.\n   - **Modelos de Classificação e Mineração de Ideias**: A tese de Luiz Fernando Spille de Souza (2021) apresenta um modelo de mineração de ideias utilizando técnicas de engenharia do conhecimento

In [66]:
result2 = agent_executor_not_memory.invoke({"input": "Realize uma análise e ofereça insights de pesquisa a partir dos trabalhos orientados pelo orientador Alexandre Leopoldo Gonçalves"})



> Entering new AgentExecutor chain...

Invoking: `qdrant_query_tool` with `{'query': 'análise e insights de pesquisa a partir dos trabalhos orientados pelo orientador Alexandre Leopoldo Gonçalves', 'query_filter': 'trabalhos orientados por Alexandre Leopoldo Gonçalves', 'orientador': ['alexandre leopoldo gonçalves']}`


análise e insights de pesquisa a partir dos trabalhos orientados pelo orientador Alexandre Leopoldo Gonçalves[]Não encontrei informações sobre trabalhos orientados pelo orientador Alexandre Leopoldo Gonçalves. Você gostaria de buscar por outro orientador ou tema relacionado a teses e dissertações?

> Finished chain.


In [70]:
result2['output']

'Não encontrei informações sobre trabalhos orientados pelo orientador Alexandre Leopoldo Gonçalves. Você gostaria de buscar por outro orientador ou tema relacionado a teses e dissertações?'

In [71]:
result3 = agent_executor_not_memory.invoke({"input": "Realize uma análise e ofereça insights de pesquisa a partir dos trabalhos orientados pelo orientador Alexandre Leopoldo Gonçalves"})



> Entering new AgentExecutor chain...

Invoking: `qdrant_query_tool` with `{'query': 'análise e insights de pesquisa a partir dos trabalhos orientados por Alexandre Leopoldo Gonçalves', 'query_filter': 'trabalhos orientados por Alexandre Leopoldo Gonçalves', 'orientador': ['alexandre leopoldo goncalves']}`


análise e insights de pesquisa a partir dos trabalhos orientados por Alexandre Leopoldo Gonçalves[ScoredPoint(id=8, version=0, score=0.4151904, payload={'autor': 'ronnie carlos tavares nunes', 'titulo': 'um modelo de perfil de aluno voltado a aplicações de técnicas de learning analytics', 'tipo': 'dissertação', 'area_de_concentracao': 'engenharia do conhecimento', 'ano_de_publicacao': 2019, 'local': 'florianópolis', 'orientador': 'alexandre leopoldo goncalves', 'coorientador': 'joao artur de souza', 'resumo': 'a análise das interações dos alunos com os ambientes virtuais de aprendizagem assumiu um papel relevante para decisões educacionais. a grande disponibilidade de cursos a di

In [72]:
result3['output']

'A análise dos trabalhos orientados por Alexandre Leopoldo Gonçalves revela uma forte ênfase em temas relacionados à Engenharia do Conhecimento, com foco em técnicas de aprendizado de máquina, mineração de dados e análise de informações em contextos educacionais e tecnológicos. Aqui estão alguns insights e tendências observadas:\n\n1. **Learning Analytics e Perfis de Alunos**:\n   - A dissertação de Ronnie Carlos Tavares Nunes propõe um modelo de perfil de aluno voltado para a aplicação de técnicas de learning analytics em ambientes de aprendizagem online. O trabalho destaca a importância de personalizar a experiência de aprendizado com base em dados coletados das interações dos alunos, sugerindo que a análise de agrupamentos e sistemas de recomendação podem melhorar a tomada de decisões educacionais.\n\n2. **Crowdsourcing e Recomendação de Trabalhadores**:\n   - Thales do Nascimento da Silva desenvolveu um modelo de recomendação de trabalhadores para tarefas em ambientes de crowdsourc

In [ ]:
result4 = agent_executor_not_memory.invoke({"input": "Realize uma análise e ofereça insights de pesquisa ca partir dos trabalhos orientados pelo orientador Alexandre Leopoldo Gonçalves nos anos de 2021, 2022 e 2023"})



> Entering new AgentExecutor chain...

Invoking: `qdrant_query_tool` with `{'query': 'análise e insights de pesquisa a partir dos trabalhos orientados por alexandre leopoldo gonçalves nos anos de 2021, 2022 e 2023', 'query_filter': 'trabalhos orientados por alexandre leopoldo gonçalves 2021 2022 2023', 'orientador': ['alexandre leopoldo goncalves'], 'ano_de_publicacao': [2021, 2022, 2023]}`


análise e insights de pesquisa a partir dos trabalhos orientados por alexandre leopoldo gonçalves nos anos de 2021, 2022 e 2023[ScoredPoint(id=27, version=0, score=0.37095264, payload={'autor': 'thales do nascimento da silva', 'titulo': 'um modelo de recomendação de trabalhadores voltado à execução de tarefas no cenário de crowdsourcing', 'tipo': 'tese', 'area_de_concentracao': 'engenharia do conhecimento', 'ano_de_publicacao': 2023, 'local': 'florianópolis', 'orientador': 'alexandre leopoldo goncalves', 'coorientador': 'joao artur de souza', 'resumo': 'atualmente o conceito de crowdsourcing vem

In [76]:
result4['output']


'A análise dos trabalhos orientados por Alexandre Leopoldo Gonçalves nos anos de 2021, 2022 e 2023 revela uma diversidade de temas e abordagens na área de Engenharia do Conhecimento. Aqui estão alguns insights e contribuições significativas de cada trabalho:\n\n1. **Thales do Nascimento da Silva - "Um modelo de recomendação de trabalhadores voltado à execução de tarefas no cenário de crowdsourcing" (2023)**:\n   - **Contribuição**: Proposta de um modelo que recomenda trabalhadores com base em suas competências para tarefas complexas em ambientes de crowdsourcing. O modelo utiliza similaridade vetorial e grafos de conhecimento, alcançando uma acurácia de 84,33% em suas avaliações.\n   - **Insight**: A pesquisa destaca a importância da inteligência coletiva e da seleção adequada de trabalhadores em cenários que exigem conhecimento especializado, contribuindo para a eficiência em processos de crowdsourcing.\n\n2. **Letícia Silveira Artese - "Modelo de descoberta de conhecimento em texto p

In [77]:
result5 = agent_executor_not_memory.invoke({"input": "Relacione e analise os pricipais conceitos que interconectam as teses das áreas de conhecimento da engenharia do conhecimento e gestão do conhecimento nos anos de 2021 e 2023"})



> Entering new AgentExecutor chain...

Invoking: `qdrant_query_tool` with `{'query': 'relacione e analise os principais conceitos que interconectam as teses das áreas de conhecimento da engenharia do conhecimento e gestão do conhecimento nos anos de 2021 e 2023', 'query_filter': 'engenharia do conhecimento e gestao do conhecimento', 'ano_de_publicacao': [2021, 2023]}`


relacione e analise os principais conceitos que interconectam as teses das áreas de conhecimento da engenharia do conhecimento e gestão do conhecimento nos anos de 2021 e 2023[ScoredPoint(id=143, version=2, score=0.59280455, payload={'autor': 'juliano keller alvez', 'titulo': 'framework adaptativo de gestão do conhecimento para a aplicação da iso 30401', 'tipo': 'tese', 'area_de_concentracao': 'gestão do conhecimento', 'ano_de_publicacao': 2023, 'local': 'florianópolis', 'orientador': 'edis mafra lapolli', 'coorientador': 'neri dos santos', 'resumo': 'a iso 30401 é a norma internacional para sistemas de gestão do conh

In [ ]:
# deu erro no filtro
result5['output']


'Aqui estão algumas teses e dissertações que abordam os conceitos interconectados entre as áreas de engenharia do conhecimento e gestão do conhecimento, publicadas entre 2021 e 2023:\n\n1. **Título:** Framework adaptativo de gestão do conhecimento para a aplicação da ISO 30401  \n   **Autor:** Juliano Keller Alvez  \n   **Ano:** 2023  \n   **Resumo:** A pesquisa desenvolve um framework adaptativo para a aplicação da norma ISO 30401, que é a norma internacional para sistemas de gestão do conhecimento. O estudo propõe um passo a passo estruturado para a implementação da gestão do conhecimento nas organizações, visando assegurar a continuidade e melhoria dos sistemas de gestão do conhecimento.  \n   **Palavras-chave:** Gestão do conhecimento, sistemas de gestão do conhecimento, ISO 30401, framework.\n\n2. **Título:** Modelo conceitual para formulação de diretrizes estratégicas na concepção e atualização de cursos de engenharia no contexto da transformação digital  \n   **Autor:** Ricardo 

In [84]:
result6 = agent_executor_not_memory.invoke({"input": "Relacione e analise os pricipais conceitos que interconectam as teses das áreas de conhecimento da engenharia do conhecimento e gestão do conhecimento nos anos de 2021 e 2023"})




> Entering new AgentExecutor chain...

Invoking: `qdrant_query_tool` with `{'query': 'relacione e analise os principais conceitos que interconectam as teses das áreas de conhecimento da engenharia do conhecimento e gestão do conhecimento nos anos de 2021 e 2023', 'query_filter': 'relacione e analise os principais conceitos que interconectam as teses das áreas de conhecimento da engenharia do conhecimento e gestão do conhecimento', 'ano_de_publicacao': [2021, 2023], 'area_de_concentracao': ['engenharia do conhecimento', 'gestão do conhecimento']}`


relacione e analise os principais conceitos que interconectam as teses das áreas de conhecimento da engenharia do conhecimento e gestão do conhecimento nos anos de 2021 e 2023[ScoredPoint(id=5, version=0, score=0.6070684, payload={'autor': 'josé tadeu silva', 'titulo': 'análise da contribuição da engenharia do conhecimento ao comércio eletrônico', 'tipo': 'dissertação', 'area_de_concentracao': 'engenharia do conhecimento', 'ano_de_publicac

In [ ]:
result6['output']

'Aqui estão algumas teses e dissertações que abordam conceitos interconectados nas áreas de Engenharia do Conhecimento e Gestão do Conhecimento, publicadas entre 2021 e 2023:\n\n1. **Título:** Análise da contribuição da engenharia do conhecimento ao comércio eletrônico  \n   **Autor:** José Tadeu Silva  \n   **Tipo:** Dissertação  \n   **Ano de Publicação:** 2022  \n   **Resumo:** A dissertação aborda como a engenharia do conhecimento pode ser aplicada para melhorar o comércio eletrônico, propondo um modelo de análise que relaciona ferramentas da engenharia do conhecimento a processos críticos no comércio eletrônico.  \n   **Palavras-chave:** Comércio eletrônico, modelo de análise, engenharia do conhecimento.\n\n2. **Título:** Modelo conceitual para formulação de diretrizes estratégicas na concepção e atualização de cursos de engenharia no contexto da transformação digital  \n   **Autor:** Ricardo Alexandre Diogo  \n   **Tipo:** Tese  \n   **Ano de Publicação:** 2023  \n   **Resumo:** 

In [88]:
result7 = agent_executor_not_memory.invoke({"input": "Analise temporalmente a evolução da área de engenharia do conhecimento tendo como base os assuntos de machine learning, inteligência artificial e oustros relacionados a estes."})




> Entering new AgentExecutor chain...

Invoking: `qdrant_query_tool` with `{'query': 'analise temporalmente a evolucao da area de engenharia do conhecimento tendo como base os assuntos de machine learning, inteligencia artificial e outros relacionados a estes', 'query_filter': 'engenharia do conhecimento, machine learning, inteligencia artificial'}`


analise temporalmente a evolucao da area de engenharia do conhecimento tendo como base os assuntos de machine learning, inteligencia artificial e outros relacionados a estes[ScoredPoint(id=19, version=0, score=0.58503115, payload={'autor': 'kaster, gerson bovi', 'titulo': 'framework conceitual baseado em aprendizagem de máquina supervisionada para concepção de sistemas judiciais de agentes inteligentes', 'tipo': 'dissertação', 'area_de_concentracao': 'engenharia do conhecimento', 'ano_de_publicacao': 2021, 'local': 'florianópolis', 'orientador': 'aires jose rover', 'coorientador': 'nao disponivel', 'resumo': 'atualmente existem várias p

In [89]:
result7['output']

'Aqui estão algumas teses e dissertações que analisam a evolução da área de engenharia do conhecimento, com foco em machine learning, inteligência artificial e temas relacionados:\n\n1. **Título:** Framework conceitual baseado em aprendizagem de máquina supervisionada para concepção de sistemas judiciais de agentes inteligentes  \n   **Autor:** Gerson Bovi Kaster  \n   **Tipo:** Dissertação  \n   **Ano de Publicação:** 2021  \n   **Resumo:** O trabalho aborda a aplicação de inteligência artificial em sistemas de informação, propondo um framework para a concepção de um sistema de agentes inteligentes na área jurídica, utilizando técnicas de aprendizagem de máquina.  \n   **Palavras-chave:** sistemas judiciais, agentes inteligentes, aprendizagem de máquina, engenharia do conhecimento.\n\n2. **Título:** Assistente virtual baseado em conhecimento no apoio a estudantes de curso de pós-graduação  \n   **Autor:** Lodacir Rodrigo Silva da Rosa  \n   **Tipo:** Dissertação  \n   **Ano de Publica